# O1 Validation & Screening — Kickoff Notebook

**Owner:** SIN-WEI SU  
**Objective:** Build the first reproducible O1 pipeline for:
1. validating RYA unreliability matrices,
2. understanding the existing instance dataset, and
3. preparing pre-inference features for the Gamma-compatibility detector.

This notebook intentionally does **not** create Gamma-compatible / incompatible labels yet. Those labels depend on the team's simulation / fit-quality work.


## 0. File setup

Update these paths if you run the notebook outside ChatGPT. Keep the original RYA files unchanged; work from copies or a project data folder.


In [5]:
from pathlib import Path
import sys
import pandas as pd
import numpy as np
import json

# Main folder where you saved all RYA project files
BASE_DIR = Path(r"C:\AIML Y2 S2\Capstone 1\RYA")

# File paths
MATRICES_JSON = BASE_DIR / "unreliability-matrices.json"
INSTANCE_ZIP = BASE_DIR / "instance_data.zip"

# Add the project folder so Python can import the helper .py files
sys.path.append(str(BASE_DIR))

# Import helper functions
from matrix_validator import validate_json
from instance_loader import load_instance_zip, quality_summary
from feature_extractor import augment_instance_dataframe, parse_r_c_vector

print("BASE_DIR:", BASE_DIR)
print("Matrices file exists:", MATRICES_JSON.exists())
print("Instance ZIP exists:", INSTANCE_ZIP.exists())


BASE_DIR: C:\AIML Y2 S2\Capstone 1\RYA
Matrices file exists: True
Instance ZIP exists: True


## 1. O1-01 — Validate the real unreliability matrices

The current approved scope is limited to **4-option Likert items**.  
We therefore separate valid 4×4 matrices from supplied matrices of other dimensions rather than silently discarding them.


In [7]:
matrix_report = validate_json(
    MATRICES_JSON,
    expected_dim=4,
    tol=0.02
)

print("Total supplied matrices:", len(matrix_report))

print("\nDimensions:")
print(matrix_report.groupby(["rows", "cols"]).size())

print("\nStatus:")
print(matrix_report["status"].value_counts())

matrix_report.head(10)


Total supplied matrices: 59

Dimensions:
rows  cols
4     4       53
6     6        6
dtype: int64

Status:
status
PASS            52
OUT_OF_SCOPE     6
WARNING          1
Name: count, dtype: int64


,item,rows,cols,in_scope_4x4,finite,non_negative,row_stochastic,max_row_sum_error,mean_diagonal,min_diagonal,mean_offdiag_mass,asymmetry,all_rows_diagonal_mode,diagonal_mode_fail_rows,status
0,chs1,6,6,False,True,True,True,0.01,0.590000,0.47,0.410000,0.001844,True,,OUT_OF_SCOPE
1,chs2,6,6,False,True,True,True,0.01,0.490000,0.38,0.510000,0.004617,True,,OUT_OF_SCOPE
2,chs3,6,6,False,True,True,True,0.01,0.596667,0.46,0.403333,0.004267,True,,OUT_OF_SCOPE
3,chs4,6,6,False,True,True,True,0.01,0.531667,0.42,0.468333,0.006656,True,,OUT_OF_SCOPE
4,chs5,6,6,False,True,True,True,0.01,0.558333,0.47,0.441667,0.004444,True,,OUT_OF_SCOPE
5,chs6,6,6,False,True,True,True,0.01,0.571667,0.45,0.428333,0.004128,True,,OUT_OF_SCOPE
6,cop2,4,4,True,True,True,True,0.01,0.545000,0.50,0.455000,0.009388,True,,PASS
7,cop3,4,4,True,True,True,True,0.01,0.582500,0.47,0.417500,0.015675,True,,PASS
8,cop4,4,4,True,True,True,True,0.01,0.660000,0.60,0.340000,0.006088,True,,PASS
9,cyb1,4,4,True,True,True,True,0.01,0.765000,0.62,0.235000,0.004087,True,,PASS


In [9]:
matrix_report[matrix_report["status"] != "PASS"][
    [
        "item",
        "rows",
        "cols",
        "status",
        "max_row_sum_error",
        "mean_diagonal",
        "min_diagonal",
        "asymmetry",
        "all_rows_diagonal_mode",
        "diagonal_mode_fail_rows"
    ]
]

,item,rows,cols,status,max_row_sum_error,mean_diagonal,min_diagonal,asymmetry,all_rows_diagonal_mode,diagonal_mode_fail_rows
0,chs1,6,6,OUT_OF_SCOPE,0.01,0.590000,0.47,0.001844,True,
1,chs2,6,6,OUT_OF_SCOPE,0.01,0.490000,0.38,0.004617,True,
2,chs3,6,6,OUT_OF_SCOPE,0.01,0.596667,0.46,0.004267,True,
3,chs4,6,6,OUT_OF_SCOPE,0.01,0.531667,0.42,0.006656,True,
4,chs5,6,6,OUT_OF_SCOPE,0.01,0.558333,0.47,0.004444,True,
5,chs6,6,6,OUT_OF_SCOPE,0.01,0.571667,0.45,0.004128,True,
58,cop1,4,4,WARNING,0.01,0.472500,0.33,0.020300,False,2


In [11]:
matrix_report.to_csv(
    BASE_DIR / "O1_matrix_validation_report.csv",
    index=False
)

print("Saved successfully.")

Saved successfully.


### What to record

Do not treat a diagonal-mode warning as automatically "bad".  
For example, an empirical retest matrix may be statistically valid even if one row's largest probability is adjacent to the diagonal. For O1, this should initially be a **feature / warning**, not an automatic rejection rule unless the team later justifies that rule.


## 2. O1-03 — load and inspect the ~63K instance dataset.


In [11]:
instances = load_instance_zip(INSTANCE_ZIP)

print("Combined shape:", instances.shape)

print("\nSource files:")
print(instances["source_file"].value_counts())

print("\nColumns:")
print(instances.columns.tolist())

Combined shape: (63328, 29)

Source files:
source_file
orthog_B_1_feature_output_df.csv                  29684
orthog_B_4_feature_output_df.csv                  29644
orthog_B_3_large_classes_feature_output_df.csv     3000
orthog_B_2_feature_output_df.csv                   1000
Name: count, dtype: int64

Columns:
['exp_code', 'X_1', 'X_2', 'X_3', 'X_4', 'Y_1', 'Y_2', 'Y_3', 'Y_4', 'XmY_1', 'XmY_2', 'XmY_3', 'XmY_4', 'mean_X', 'stdev_X', 'skewness_X', 'kurtosis_X', 'mean_Y', 'stdev_Y', 'skewness_Y', 'kurtosis_Y', 'euclid_dist_XY', 'n_X', 'n_Y', 'P_matrix', 'P_matrix_question', 'shape', 'rate', 'source_file']


In [13]:
quality = quality_summary(instances)

quality.head(20)

,dtype,missing,missing_pct,unique
skewness_X,float64,34,0.053689,57350
kurtosis_Y,float64,34,0.053689,55333
skewness_Y,float64,34,0.053689,57314
kurtosis_X,float64,34,0.053689,55286
exp_code,object,0,0.000000,63328
rate,float64,0,0.000000,63328
shape,float64,0,0.000000,63328
P_matrix_question,object,0,0.000000,53
P_matrix,object,0,0.000000,63328
n_Y,int64,0,0.000000,3156


In [15]:
missing_moments = instances[
    instances[
        ["skewness_X", "kurtosis_X", "skewness_Y", "kurtosis_Y"]
    ].isna().any(axis=1)
]

print("Rows with missing moment features:", len(missing_moments))

missing_moments[
    [
        "exp_code",
        "P_matrix_question",
        "X_1", "X_2", "X_3", "X_4",
        "Y_1", "Y_2", "Y_3", "Y_4",
        "n_X", "n_Y",
        "skewness_X", "kurtosis_X",
        "skewness_Y", "kurtosis_Y",
        "shape", "rate",
        "source_file"
    ]
].head(20)

Rows with missing moment features: 68


,exp_code,P_matrix_question,X_1,X_2,X_3,X_4,Y_1,Y_2,Y_3,Y_4,n_X,n_Y,skewness_X,kurtosis_X,skewness_Y,kurtosis_Y,shape,rate,source_file
3435,orthog_B_4_443,ry21,0.548387,0.064516,0.225806,0.161290,0.000000,0.000000,0.000000,1.000000,31,5,0.572299,1.665289,NaN,NaN,6.217545,0.257431,orthog_B_4_feature_output_df.csv
5960,orthog_B_4_2993,ry2,0.108696,0.043478,0.347826,0.500000,0.000000,1.000000,0.000000,0.000000,46,5,-1.230216,3.540291,NaN,NaN,4.847563,0.363600,orthog_B_4_feature_output_df.csv
6297,orthog_B_4_3335,ry10,0.000000,1.000000,0.000000,0.000000,0.432749,0.418129,0.149123,0.000000,5,342,NaN,NaN,0.467074,2.074258,1.800876,0.235006,orthog_B_4_feature_output_df.csv
7415,orthog_B_4_4462,sun4,0.191011,0.174157,0.348315,0.286517,1.000000,0.000000,0.000000,0.000000,178,5,-0.375399,1.890026,NaN,NaN,4.384870,0.369193,orthog_B_4_feature_output_df.csv
7778,orthog_B_4_4829,ry17,0.311688,0.142857,0.142857,0.402597,0.000000,0.000000,1.000000,0.000000,77,5,-0.175245,1.325634,NaN,NaN,4.222697,0.375043,orthog_B_4_feature_output_df.csv
8030,orthog_B_4_5082,ry25,0.000000,0.000000,1.000000,0.000000,0.000000,0.317406,0.430034,0.252560,5,293,NaN,NaN,0.106895,1.772881,2.399939,0.494896,orthog_B_4_feature_output_df.csv
8550,orthog_B_4_5611,ry26,0.000000,1.000000,0.000000,0.000000,0.217391,0.000000,0.347826,0.434783,7,23,NaN,NaN,-0.875595,2.300000,3.666360,0.318636,orthog_B_4_feature_output_df.csv
11909,orthog_B_4_9006,ry29,0.000000,0.000000,1.000000,0.000000,0.064516,0.274194,0.648387,0.012903,6,930,NaN,NaN,-1.047907,3.462632,2.079672,0.559988,orthog_B_4_feature_output_df.csv
12390,orthog_B_4_9497,cop3,0.000000,0.000000,1.000000,0.000000,0.293801,0.293801,0.156334,0.256065,6,371,NaN,NaN,0.233019,1.610373,3.704554,0.408610,orthog_B_4_feature_output_df.csv
12450,orthog_B_4_9557,cop4,0.204918,0.483607,0.008197,0.303279,0.000000,0.000000,1.000000,0.000000,122,5,0.434755,1.745390,NaN,NaN,4.026717,0.386953,orthog_B_4_feature_output_df.csv


In [17]:
# Check whether missing moments are associated with zero-variance / one-category histograms

missing_check = missing_moments.copy()

missing_check["active_categories_X"] = (
    missing_check[["X_1", "X_2", "X_3", "X_4"]] > 0
).sum(axis=1)

missing_check["active_categories_Y"] = (
    missing_check[["Y_1", "Y_2", "Y_3", "Y_4"]] > 0
).sum(axis=1)

print("Active categories in X:")
print(missing_check["active_categories_X"].value_counts().sort_index())

print("\nActive categories in Y:")
print(missing_check["active_categories_Y"].value_counts().sort_index())

Active categories in X:
active_categories_X
1    34
3     1
4    33
Name: count, dtype: int64

Active categories in Y:
active_categories_Y
1    34
2     1
3     4
4    29
Name: count, dtype: int64


In [19]:
print("Rows where X has exactly one active category:",
      (missing_check["active_categories_X"] == 1).sum())

print("Rows where Y has exactly one active category:",
      (missing_check["active_categories_Y"] == 1).sum())

print("Total rows with missing moments:",
      len(missing_check))

Rows where X has exactly one active category: 34
Rows where Y has exactly one active category: 34
Total rows with missing moments: 68


In [21]:
one_hot_X = missing_check["active_categories_X"] == 1
one_hot_Y = missing_check["active_categories_Y"] == 1

print("X or Y has one active category:", (one_hot_X | one_hot_Y).sum())
print("Both X and Y have one active category:", (one_hot_X & one_hot_Y).sum())
print("Neither X nor Y has one active category:", (~(one_hot_X | one_hot_Y)).sum())

X or Y has one active category: 68
Both X and Y have one active category: 0
Neither X nor Y has one active category: 0


In [23]:
summary_cols = [
    "n_X",
    "n_Y",
    "euclid_dist_XY",
    "shape",
    "rate"
]

instances[summary_cols].describe(
    percentiles=[0.01, 0.05, 0.25, 0.5, 0.75, 0.95, 0.99]
).T

,count,mean,std,min,1%,5%,25%,50%,75%,95%,99%,max
n_X,63328.0,402.788924,814.319337,5.000000,6.000000,11.000000,37.000000,199.000000,458.000000,976.000000,4873.190000,9819.000000
n_Y,63328.0,406.541798,827.153358,1.000000,6.000000,11.000000,38.000000,202.000000,459.000000,976.000000,4924.460000,9940.000000
euclid_dist_XY,63328.0,0.395504,0.168776,0.007900,0.084800,0.145800,0.272300,0.380500,0.501200,0.697965,0.849200,1.388200
shape,63328.0,12.178270,38.650495,0.669738,1.359936,1.541177,2.064503,3.492693,8.911540,44.531114,145.026893,2630.171626
rate,63328.0,0.271487,0.071383,0.014548,0.122764,0.175640,0.234679,0.258361,0.296467,0.409049,0.495303,1.099945


In [25]:
sample = instances.head(100).copy()

sample_aug = augment_instance_dataframe(sample)

new_cols = [
    c for c in sample_aug.columns
    if c not in instances.columns
]

print("Number of new O1 features:", len(new_cols))
print(new_cols)

sample_aug[new_cols].head()

Number of new O1 features: 43
['log_n_X', 'log_n_Y', 'group_size_ratio', 'log_group_size_ratio', 'min_prop_X', 'min_prop_Y', 'max_prop_X', 'max_prop_Y', 'active_categories_X', 'active_categories_Y', 'degenerate_X', 'degenerate_Y', 'sparse_count_X', 'sparse_count_Y', 'entropy_X', 'entropy_Y', 'mean_diag_P', 'min_diag_P', 'max_diag_P', 'mean_offdiag_mass_P', 'asymmetry_P', 'max_row_sum_error_P', 'all_rows_diagonal_mode_P', 'absdiff_1', 'absdiff_2', 'absdiff_3', 'absdiff_4', 'P_11', 'P_12', 'P_13', 'P_14', 'P_21', 'P_22', 'P_23', 'P_24', 'P_31', 'P_32', 'P_33', 'P_34', 'P_41', 'P_42', 'P_43', 'P_44']


,log_n_X,log_n_Y,group_size_ratio,log_group_size_ratio,min_prop_X,min_prop_Y,max_prop_X,max_prop_Y,active_categories_X,active_categories_Y,...,P_23,P_24,P_31,P_32,P_33,P_34,P_41,P_42,P_43,P_44
0,8.469263,8.011355,1.580956,0.458029,0.182371,0.084937,0.330325,0.424021,4,4,...,0.215,0.064,0.030,0.066,0.736,0.168,0.023,0.021,0.081,0.875
1,6.689599,7.926603,3.448319,1.237887,0.113325,0.046226,0.357410,0.417118,4,4,...,0.416,0.127,0.043,0.172,0.666,0.119,0.105,0.078,0.286,0.531
2,8.464847,7.354362,3.037132,1.110914,0.066400,0.168374,0.410413,0.327145,4,4,...,0.245,0.037,0.088,0.130,0.642,0.140,0.082,0.041,0.149,0.728
3,7.846199,6.755769,2.977855,1.091203,0.008219,0.137529,0.457926,0.340326,4,4,...,0.159,0.048,0.013,0.156,0.698,0.133,0.012,0.018,0.128,0.841
4,6.297109,8.895356,13.463100,2.599953,0.075646,0.196930,0.512915,0.373441,4,4,...,0.276,0.106,0.015,0.120,0.602,0.263,0.018,0.024,0.157,0.801


In [27]:
instances_aug = augment_instance_dataframe(instances)

print("Original shape:", instances.shape)
print("Augmented shape:", instances_aug.shape)

print("\nAny duplicated column names?")
print(instances_aug.columns.duplicated().sum())

print("\nTotal missing values after augmentation:")
print(instances_aug.isna().sum().sum())

Original shape: (63328, 29)
Augmented shape: (63328, 72)

Any duplicated column names?
0

Total missing values after augmentation:
136


In [25]:
OUTPUT_FILE = BASE_DIR / "O1_instance_features.csv"

instances_aug.to_csv(
    OUTPUT_FILE,
    index=False
)

print("Saved to:")
print(OUTPUT_FILE)

Saved to:
C:\AIML Y2 S2\Capstone 1\RYA\O1_instance_features.csv


In [29]:
print("Top 20 most extreme group-size ratios:")

instances_aug.nlargest(
    20,
    "group_size_ratio"
)[
    [
        "exp_code",
        "P_matrix_question",
        "n_X",
        "n_Y",
        "group_size_ratio",
        "active_categories_X",
        "active_categories_Y",
        "degenerate_X",
        "degenerate_Y",
        "sparse_count_X",
        "sparse_count_Y",
        "euclid_dist_XY",
        "mean_diag_P",
        "asymmetry_P",
        "shape",
        "rate",
        "source_file"
    ]
]

Top 20 most extreme group-size ratios:


,exp_code,P_matrix_question,n_X,n_Y,group_size_ratio,active_categories_X,active_categories_Y,degenerate_X,degenerate_Y,sparse_count_X,sparse_count_Y,euclid_dist_XY,mean_diag_P,asymmetry_P,shape,rate,source_file
2789,orthog_B_3_large_classes_2790,ry2,3337,1,3337.000000,4,3,0,0,1,1,1.3882,0.66150,0.016563,1.779008,0.140686,orthog_B_3_large_classes_feature_output_df.csv
32839,orthog_B_2_large_classes_196,ry12,8533,7,1219.000000,4,3,0,0,1,1,0.3302,0.61750,0.013410,1.893022,0.528730,orthog_B_2_feature_output_df.csv
33609,orthog_B_2_large_classes_966,ry25,6927,6,1154.500000,4,4,0,0,0,0,0.3681,0.68975,0.003086,1.436196,0.316948,orthog_B_2_feature_output_df.csv
398,orthog_B_3_large_classes_399,ry30,7160,7,1022.857143,4,4,0,0,0,0,0.2119,0.65700,0.006432,1.238826,0.379772,orthog_B_3_large_classes_feature_output_df.csv
2165,orthog_B_3_large_classes_2166,ry24,6,5215,869.166667,4,4,0,0,0,0,0.3840,0.58900,0.017971,2.466704,0.511242,orthog_B_3_large_classes_feature_output_df.csv
475,orthog_B_3_large_classes_476,for1,7635,10,763.500000,4,4,0,0,0,0,0.2246,0.62225,0.009012,1.311593,0.317466,orthog_B_3_large_classes_feature_output_df.csv
737,orthog_B_3_large_classes_738,phq4,4589,7,655.571429,4,2,0,0,0,2,0.9553,0.66925,0.006376,5.788921,0.261440,orthog_B_3_large_classes_feature_output_df.csv
1161,orthog_B_3_large_classes_1162,ry20,4435,7,633.571429,4,3,0,0,0,1,0.2630,0.66675,0.006024,1.715217,0.480912,orthog_B_3_large_classes_feature_output_df.csv
33479,orthog_B_2_large_classes_836,ry4,9,5685,631.666667,3,4,0,0,1,1,0.6497,0.58125,0.007307,2.659586,0.322014,orthog_B_2_feature_output_df.csv
2032,orthog_B_3_large_classes_2033,ry8,6,3541,590.166667,3,4,0,0,1,0,0.6183,0.57475,0.018689,3.848754,0.413576,orthog_B_3_large_classes_feature_output_df.csv


In [31]:
print("Degenerate histogram cases:")

degenerate_cases = instances_aug[
    (instances_aug["degenerate_X"] == 1) |
    (instances_aug["degenerate_Y"] == 1)
]

print("Total degenerate cases:", len(degenerate_cases))

degenerate_cases[
    [
        "exp_code",
        "P_matrix_question",
        "n_X",
        "n_Y",
        "group_size_ratio",
        "active_categories_X",
        "active_categories_Y",
        "degenerate_X",
        "degenerate_Y",
        "euclid_dist_XY",
        "shape",
        "rate",
        "source_file"
    ]
].head(20)

Degenerate histogram cases:
Total degenerate cases: 68


,exp_code,P_matrix_question,n_X,n_Y,group_size_ratio,active_categories_X,active_categories_Y,degenerate_X,degenerate_Y,euclid_dist_XY,shape,rate,source_file
3435,orthog_B_4_443,ry21,31,5,6.200000,4,1,0,1,1.0292,6.217545,0.257431,orthog_B_4_feature_output_df.csv
5960,orthog_B_4_2993,ry2,46,5,9.200000,4,1,0,1,1.1392,4.847563,0.363600,orthog_B_4_feature_output_df.csv
6297,orthog_B_4_3335,ry10,5,342,68.400000,1,3,1,0,0.7403,1.800876,0.235006,orthog_B_4_feature_output_df.csv
7415,orthog_B_4_4462,sun4,178,5,35.600000,4,1,0,1,0.9424,4.384870,0.369193,orthog_B_4_feature_output_df.csv
7778,orthog_B_4_4829,ry17,77,5,15.400000,4,1,0,1,1.0071,4.222697,0.375043,orthog_B_4_feature_output_df.csv
8030,orthog_B_4_5082,ry25,5,293,58.600000,1,3,1,0,0.6996,2.399939,0.494896,orthog_B_4_feature_output_df.csv
8550,orthog_B_4_5611,ry26,7,23,3.285714,1,3,1,0,1.1650,3.666360,0.318636,orthog_B_4_feature_output_df.csv
11909,orthog_B_4_9006,ry29,6,930,155.000000,1,4,1,0,0.4507,2.079672,0.559988,orthog_B_4_feature_output_df.csv
12390,orthog_B_4_9497,cop3,6,371,61.833333,1,4,1,0,0.9747,3.704554,0.408610,orthog_B_4_feature_output_df.csv
12450,orthog_B_4_9557,cop4,122,5,24.400000,4,1,0,1,1.1625,4.026717,0.386953,orthog_B_4_feature_output_df.csv


In [33]:
edge_summary = pd.DataFrame({
    "condition": [
        "Degenerate X",
        "Degenerate Y",
        "Either degenerate",
        "Group ratio >= 10",
        "Group ratio >= 50",
        "Group ratio >= 100",
        "X has <= 2 active categories",
        "Y has <= 2 active categories",
        "Either group has <= 2 active categories",
        "Any P matrix not diagonal-mode"
    ],
    "count": [
        (instances_aug["degenerate_X"] == 1).sum(),
        (instances_aug["degenerate_Y"] == 1).sum(),
        ((instances_aug["degenerate_X"] == 1) |
         (instances_aug["degenerate_Y"] == 1)).sum(),
        (instances_aug["group_size_ratio"] >= 10).sum(),
        (instances_aug["group_size_ratio"] >= 50).sum(),
        (instances_aug["group_size_ratio"] >= 100).sum(),
        (instances_aug["active_categories_X"] <= 2).sum(),
        (instances_aug["active_categories_Y"] <= 2).sum(),
        ((instances_aug["active_categories_X"] <= 2) |
         (instances_aug["active_categories_Y"] <= 2)).sum(),
        (instances_aug["all_rows_diagonal_mode_P"] == 0).sum()
    ]
})

edge_summary["percent"] = (
    edge_summary["count"] / len(instances_aug) * 100
)

edge_summary

,condition,count,percent
0,Degenerate X,34,0.053689
1,Degenerate Y,34,0.053689
2,Either degenerate,68,0.107377
3,Group ratio >= 10,19307,30.487304
4,Group ratio >= 50,2708,4.276150
5,Group ratio >= 100,621,0.980609
6,X has <= 2 active categories,883,1.394328
7,Y has <= 2 active categories,889,1.403802
8,Either group has <= 2 active categories,1765,2.787077
9,Any P matrix not diagonal-mode,1185,1.871210


In [35]:
print("Degenerate cases by source file:")
print(
    degenerate_cases["source_file"]
    .value_counts()
)

print("\nDegenerate cases by P_matrix_question:")
print(
    degenerate_cases["P_matrix_question"]
    .value_counts()
    .head(15)
)

Degenerate cases by source file:
source_file
orthog_B_4_feature_output_df.csv    35
orthog_B_1_feature_output_df.csv    33
Name: count, dtype: int64

Degenerate cases by P_matrix_question:
P_matrix_question
ry28    5
ry30    5
ry21    3
ry29    3
ry3     3
ry7     3
cop3    3
ry12    3
ry17    3
ry10    2
sun4    2
for1    2
ry24    2
cop4    2
ry6     2
Name: count, dtype: int64


## 3. O1-02 — Create screening-specific pre-inference features

These features must be available **before** the current Gemfinder neural network is used.

Do **not** use `shape` or `rate` as detector input features: they are Gamma-fit outputs/targets, not pre-inference information.


In [37]:
# ---------------------------------------------------------
# O1-02: Define classifier-ready feature groups
# ---------------------------------------------------------

identifier_cols = [
    "exp_code",
    "P_matrix_question",
    "source_file"
]

# Gamma fitting outputs: keep for reference only,
# but DO NOT use them as O1 classifier inputs.
excluded_output_cols = [
    "shape",
    "rate"
]

screening_features = [
    # Group size
    "n_X",
    "n_Y",
    "log_n_X",
    "log_n_Y",
    "group_size_ratio",
    "log_group_size_ratio",

    # X distribution
    "X_1",
    "X_2",
    "X_3",
    "X_4",

    # Y distribution
    "Y_1",
    "Y_2",
    "Y_3",
    "Y_4",

    # X-Y difference
    "euclid_dist_XY",
    "absdiff_1",
    "absdiff_2",
    "absdiff_3",
    "absdiff_4",

    # Histogram structure
    "min_prop_X",
    "min_prop_Y",
    "max_prop_X",
    "max_prop_Y",
    "active_categories_X",
    "active_categories_Y",
    "degenerate_X",
    "degenerate_Y",
    "sparse_count_X",
    "sparse_count_Y",
    "entropy_X",
    "entropy_Y",

    # P-matrix structure
    "mean_diag_P",
    "min_diag_P",
    "max_diag_P",
    "mean_offdiag_mass_P",
    "asymmetry_P",
    "max_row_sum_error_P",
    "all_rows_diagonal_mode_P",

    # Raw P-matrix entries
    "P_11", "P_12", "P_13", "P_14",
    "P_21", "P_22", "P_23", "P_24",
    "P_31", "P_32", "P_33", "P_34",
    "P_41", "P_42", "P_43", "P_44"
]

print("Identifier columns:", len(identifier_cols))
print("Candidate O1 screening features:", len(screening_features))
print("Excluded Gamma outputs:", excluded_output_cols)

Identifier columns: 3
Candidate O1 screening features: 54
Excluded Gamma outputs: ['shape', 'rate']


In [39]:
X_screen = instances_aug[screening_features].copy()

print("Classifier input shape:", X_screen.shape)

print("\nMissing values:")
print(X_screen.isna().sum().sum())

print("\nInfinite values:")
print(
    np.isinf(
        X_screen.select_dtypes(include=[np.number])
    ).sum().sum()
)

print("\nDuplicated feature names:")
print(X_screen.columns.duplicated().sum())

Classifier input shape: (63328, 54)

Missing values:
0

Infinite values:
0

Duplicated feature names:
0


In [41]:
non_numeric = X_screen.select_dtypes(
    exclude=[np.number]
).columns.tolist()

print("Non-numeric classifier features:")
print(non_numeric)

Non-numeric classifier features:
[]


In [43]:
classifier_ready = pd.concat(
    [
        instances_aug[identifier_cols].copy(),
        X_screen.copy()
    ],
    axis=1
)

print("Classifier-ready shape:", classifier_ready.shape)

classifier_ready.head()

Classifier-ready shape: (63328, 57)


,exp_code,P_matrix_question,source_file,n_X,n_Y,log_n_X,log_n_Y,group_size_ratio,log_group_size_ratio,X_1,...,P_23,P_24,P_31,P_32,P_33,P_34,P_41,P_42,P_43,P_44
0,orthog_B_3_large_classes_1,cyb1,orthog_B_3_large_classes_feature_output_df.csv,4765,3014,8.469263,8.011355,1.580956,0.458029,0.182371,...,0.215,0.064,0.030,0.066,0.736,0.168,0.023,0.021,0.081,0.875
1,orthog_B_3_large_classes_2,cop1,orthog_B_3_large_classes_feature_output_df.csv,803,2769,6.689599,7.926603,3.448319,1.237887,0.229141,...,0.416,0.127,0.043,0.172,0.666,0.119,0.105,0.078,0.286,0.531
2,orthog_B_3_large_classes_3,phq5,orthog_B_3_large_classes_feature_output_df.csv,4744,1562,8.464847,7.354362,3.037132,1.110914,0.366779,...,0.245,0.037,0.088,0.130,0.642,0.140,0.082,0.041,0.149,0.728
3,orthog_B_3_large_classes_4,ry19,orthog_B_3_large_classes_feature_output_df.csv,2555,858,7.846199,6.755769,2.977855,1.091203,0.457926,...,0.159,0.048,0.013,0.156,0.698,0.133,0.012,0.018,0.128,0.841
4,orthog_B_3_large_classes_5,ry9,orthog_B_3_large_classes_feature_output_df.csv,542,7297,6.297109,8.895356,13.463100,2.599953,0.324723,...,0.276,0.106,0.015,0.120,0.602,0.263,0.018,0.024,0.157,0.801


In [45]:
CLASSIFIER_INPUT_FILE = (
    BASE_DIR / "O1_classifier_inputs_unlabelled.csv"
)

classifier_ready.to_csv(
    CLASSIFIER_INPUT_FILE,
    index=False
)

print("Saved:")
print(CLASSIFIER_INPUT_FILE)

Saved:
C:\AIML Y2 S2\Capstone 1\RYA\O1_classifier_inputs_unlabelled.csv
